# 03 — Particionamiento y plan de Spark

Compara la lectura de las interacciones por año y dos configuraciones de shuffle. Es un experimento local descriptivo: los tiempos dependen del equipo, el caché del sistema y las ejecuciones concurrentes. No modifica los datos oficiales.


In [ ]:
import platform
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


ROOT = Path.cwd().resolve()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.processing.build_fact_award import configure_pyspark_python  # noqa: E402
from src.processing.build_fact_award_item import configure_windows_hadoop  # noqa: E402

#tabla agregada por entidad, proveedor y año; sirve para probar lectura por particiones.
data = ROOT / 'data' / 'processed' / 'entity_supplier_year'
if not (data / '_SUCCESS').is_file():
    raise FileNotFoundError(f'Falta la tabla publicada: {data}')

# Configura los ejecutables locales requeridos antes de iniciar Spark.
hadoop = configure_windows_hadoop()
configure_pyspark_python()

# local[4] -> usa hasta cuatro hilos del equipo que permite una prueba paralela sin ocupar demasiados recursos.
# AQE se desactiva para que el plan no cambie automaticamente y la comparacion 4 vs. 8 sea consistente.
builder = (SparkSession.builder.appName('parte1-particionamiento')
           .master('local[4]').config('spark.sql.adaptive.enabled', 'false'))
if hadoop:
    builder = builder.config('spark.driver.extraLibraryPath', str(hadoop / 'bin'))
spark = builder.getOrCreate()

# leer solo archivos Parquet y reconoce award_year como columna de partición.
frame = spark.read.option('pathGlobFilter', '*.parquet').parquet(str(data))

# mostrar versiones y particiones disponibles para dejar claro el entorno de la prueba.
print('Sistema:', platform.platform(), '| Python:', platform.python_version(),
      '| Spark:', spark.version)
print('Master:', spark.sparkContext.master, '| AQE:', spark.conf.get('spark.sql.adaptive.enabled'))
print('Archivos Parquet:', len(list(data.rglob('*.parquet'))))
print('Particiones por año:', sorted(p.name for p in data.glob('award_year=*')))

# primer plan -> revisar toda la tabla que es la referencia sin aprovechar el particionamiento por año.
print('Plan sin filtro de año:')
frame.groupBy('entity_id').agg(F.sum('award_count').alias('total')).explain()
# eligimos 2023 porque es una partición completa y permite comprobar que Spark lea solo ese año.
print('Plan con filtro 2023:')
frame.where(F.col('award_year') == 2023).groupBy('entity_id').agg(
    F.sum('award_count').alias('total')).explain()

results = []
# probamos 4 y 8 particiones con valores pequeños, cercanos a los 4 hilos locales.
# tres repeticiones separan la primera corrida de calentamiento de dos mediciones comparables.
for partitions in (4, 8):
    spark.conf.set('spark.sql.shuffle.partitions', str(partitions))
    for repetition in range(3):
        start = time.perf_counter()
        #mantener misma consulta y filtro y solo cambiar la cantidad de particiones del shuffle.
        query = frame.where(F.col('award_year') == 2023).groupBy('entity_id').agg(
            F.sum('award_count').alias('total'))
        # first() obliga a Spark a ejecutar el cálculo antes de registrar el tiempo.
        result = query.agg(F.count('*').alias('entities'), F.sum('total').alias('awards')).first()
        results.append({'shuffle': partitions, 'repeticion': repetition,
                        'tipo': 'calentamiento' if repetition == 0 else 'medición',
                        'segundos': round(time.perf_counter() - start, 3),
                        # Estos valores deben coincidir entre filas: validan que no cambió el resultado.
                        'entidades': result.entities, 'adjudicaciones': result.awards})

# comparar tiempos y comprueba que ambas configuraciones devolvieron los mismos conteos.
display(pd.DataFrame(results))

# liberar los recursos de Spark al finalizar la prueba local.
spark.stop()


Sistema: Windows-11-10.0.26300-SP0 | Python: 3.12.10 | Spark: 3.5.6
Master: local[4] | AQE: false
Archivos Parquet: 40
Particiones por año: ['award_year=2015', 'award_year=2016', 'award_year=2017', 'award_year=2018', 'award_year=2019', 'award_year=2020', 'award_year=2021', 'award_year=2022', 'award_year=2023', 'award_year=2024']
Plan sin filtro de año:
== Physical Plan ==
*(2) HashAggregate(keys=[entity_id#0], functions=[sum(award_count#3L)])
+- Exchange hashpartitioning(entity_id#0, 200), ENSURE_REQUIREMENTS, [plan_id=24]
   +- *(1) HashAggregate(keys=[entity_id#0], functions=[partial_sum(award_count#3L)])
      +- *(1) Project [entity_id#0, award_count#3L]
         +- *(1) ColumnarToRow
            +- FileScan parquet [entity_id#0,award_count#3L,award_year#7] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/HP/Desktop/2026-02/DataMining/Proyecto2/data/processed/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<entity_

,shuffle,repeticion,tipo,segundos,entidades,adjudicaciones
0,4,0,calentamiento,2.816,2510,41759
1,4,1,medición,0.838,2510,41759
2,4,2,medición,0.459,2510,41759
3,8,0,calentamiento,0.842,2510,41759
4,8,1,medición,1.240,2510,41759
5,8,2,medición,0.783,2510,41759


Esta celda compara cómo Spark prepara y ejecuta la misma consulta con distintas configuraciones de particiones.

- **Plan sin filtro / con filtro 2023**: muestran los pasos que Spark planea ejecutar. Al filtrar por año, Spark puede leer solo la partición de 2023 en vez de revisar toda la tabla.
- **Shuffle 4 y 8**: cantidad de partes en que Spark reparte el trabajo después de agrupar. No existe un número siempre mejor; depende del volumen y del equipo.
- **Calentamiento**: primera ejecución. Puede tardar más porque Spark y el sistema preparan recursos.
- **Medición**: ejecuciones posteriores, útiles para comparar tiempos con menos efecto del inicio.
- **Entidades y adjudicaciones** deben mantenerse iguales entre filas: cambia la forma de procesar, no el resultado.

Este resultado es una prueba local de rendimiento, no una regla universal para cualquier computadora.
